# Paper 1 — EliteTable: evolving executable policies with an LLM

**Working research notebook, not a results claim.** The implementation is named
`EliteSearch`. This notebook is the intermediary between simulation outputs and
the first manuscript. The agreed three-paper sequence and full protocol are in
[README.md](README.md).

**Research question:** Can EliteTable produce policies that reach an environment's
score target, or improve the best policy's score over generations?

The primary evidence is best-so-far search reward versus generation, with the
recorded score target marked. Held-out evaluation checks the selected policies
on unseen episode seeds. Reaching a search target and reaching it on held-out
episodes are reported separately.

The LLM weights stay fixed. The system evolves Python policies through invention,
editing, remixing, evaluation and elite retention. Success on familiar Gymnasium
tasks does not establish discovery of previously unknown strategies.

**Workflow:** edit the launch settings → copy a command into a separate terminal →
let the CLI save its run directory → rerun the analysis cells here.
**Run All never starts a simulation or makes model calls.**

**Kernel:** use Python with NumPy, Matplotlib and IPython. See the README for setup;
printed simulation commands use the repository virtual environment independently.


## 1. Paper outline and claim boundaries

1. **Introduction:** executable policy search with an API-only language model.
2. **Algorithm:** fixed elite table, frozen parents per generation, 20% fresh
   proposals / 40% remixes / remainder edits after generation one, mean reward
   selection, duplicate rejection, bounded repair, termination by search target or maximum generation budget.
3. **Protocol:** model/provider, environment contracts, search and test seed panels,
   independent searches, failure handling, budgets and archived source.
4. **Results:** target attainment and best-policy improvement per environment;
   best-so-far search curves with target lines; held-out checks; policy lineages.
5. **Limitations:** prior knowledge of public tasks, overfitting to search seeds,
   sparse/noisy rewards, runtime failures, pixel observations, provider variability.
6. **Conclusion:** only claims supported by the completed runs.

Reserve systematic ablation for Paper 2 and broad comparisons for Paper 3. Before
claiming a benefit over independent sampling, add a small budget-matched control.
The current fresh-proposal prompt sees the elite summaries, so it is **not** that
control. No comparative superiority claim is supported by this notebook alone.

BipedalWalker is a proposed failure case to investigate, not a predetermined result.
Use “not solved within the tested budget/configuration” if the evidence warrants it.

In [ ]:
from pathlib import Path
import csv
import io
import json
import math
import shlex
import sqlite3
from contextlib import closing
from collections import defaultdict
from statistics import fmean

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Code, Markdown, display

# Works from the repo root, this notebook's folder, or a directory beneath either.
ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "rsikit" / "policy.py").is_file() and (p / "pyproject.toml").is_file()
)
WORK = ROOT / "examples" / "elitelist_papers"
RUNS = WORK / "runs"
FIGURES = WORK / "figures"
PYTHON = ROOT / ".venv" / "bin" / "python"
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
print(f"Read runs from: {RUNS}")

## 2. Choose experiments and print launch commands

Paper 1 uses CartPole, MountainCar, Acrobot, LunarLander, BipedalWalker and
CarRacing. The agreed budget is ten generations and ten independent searches per
environment; this replicate count is not a power calculation. Settings below
prepare the main study. Complete the preflight in PROTOCOL.md before launching.

Search seeds are reused for fair candidate ranking. The held-out panel is never
shown to the search. Hosted LLM calls are not made deterministic by `search_seed`.
Use fresh held-out seeds for the main study after inspecting pilot results.
Blackjack generally needs a much larger episode panel; default CliffWalking is
deterministic, and FrozenLake uses a fixed map despite changing transition seeds.

The environment descriptions come from the installed Gymnasium class documentation;
resolved settings are explicitly prepended and archived. LunarLander uses standard
discrete actions/no wind; continuous actions are a separate ID. Blackjack-v1 is
Gymnasium's game. CarRacing policies consume native 96×96 RGB arrays.

Paper 1 disables early stopping: every successful search runs all ten generations,
including after target attainment. Targets remain reporting markers, saved as
`reporting_target`; they do not terminate search. Set `TARGET_SCORE` only to
override the reporting threshold. A search target hit is not held-out success.
The general CLI still enables early stopping unless `--no-early-stop` is supplied.


In [ ]:
SUITES = {
    "classic_control": ["CartPole-v1", "MountainCar-v0", "MountainCarContinuous-v0", "Acrobot-v1", "Pendulum-v1"],
    "toy_text": ["Blackjack-v1", "FrozenLake-v1", "CliffWalking-v1", "Taxi-v4"],
    "box2d": ["LunarLander-v3", "BipedalWalker-v3", "CarRacing-v3"],
    "variants": ["LunarLanderContinuous-v3", "BipedalWalkerHardcore-v3", "FrozenLake8x8-v1", "CliffWalkingSlippery-v1"],
}
ENVIRONMENTS = ["CartPole-v1", "MountainCar-v0", "Acrobot-v1",
                "LunarLander-v3", "BipedalWalker-v3", "CarRacing-v3"]
# To cover the main suite: sum((SUITES[k] for k in ("classic_control", "toy_text", "box2d")), [])
EXPERIMENT = "main-v1"  # Use a new label for a new protocol; completed outputs are never overwritten.
SEARCH_SEEDS = range(10)
POPULATION, GENERATIONS, ELITES = 50, 10, 10
MAX_REPAIRS = 5
EPISODE_TIMEOUT = 10.0  # Wall-clock seconds per episode, per seed.
EARLY_STOP = False
TARGET_SCORE = None  # None uses Gymnasium's registered threshold; set a score to override.
TRAIN_SEEDS = list(range(10))
TEST_SEEDS = list(range(1000, 1100))
MODEL = "openai/gpt-oss-120b:nitro"

# Paper 1: 6 environments × 10 searches × 10 generations. See PROTOCOL.md.
# Complete the reproducibility preflight before launching this main-study batch.
commands = []
for env in ENVIRONMENTS:
    for seed in SEARCH_SEEDS:
        name = f"{EXPERIMENT}-{env}-{seed}"
        output = RUNS / name
        command = [
            str(PYTHON), "-u", "-B", "-m", "examples.elitelist_papers.run",
            "--env", env, "--model", MODEL, "--search-seed", str(seed),
            "--population", str(POPULATION), "--generations", str(GENERATIONS),
            "--elites", str(ELITES), "--max-repairs", str(MAX_REPAIRS),
            "--episode-timeout", str(EPISODE_TIMEOUT), "--generation-concurrency", "50", "--concurrency", "4",
            "--seeds", *map(str, TRAIN_SEEDS), "--heldout-seeds", *map(str, TEST_SEEDS),
            "--output", str(output),
        ]
        if TARGET_SCORE is not None:
            command += ["--target-score", str(TARGET_SCORE)]
        if not EARLY_STOP:
            command.append("--no-early-stop")
        commands.append((name, command))

# This only prints shell commands. Copy ONE into a separate terminal to start.
print(f"In your other terminal: cd {shlex.quote(str(ROOT))}")
print(f"rtk proxy mkdir -p {shlex.quote(str(WORK / 'logs'))}")
print("Ensure OPENROUTER_API_KEY is exported and the Docker worker image is available.\n")
for name, command in commands:
    log = WORK / "logs" / f"{name}.log"
    shell = f"nohup {shlex.join(command)} >> {shlex.quote(str(log))} 2>&1 < /dev/null & echo $!"
    print("rtk proxy sh -c " + shlex.quote(shell))
    print(f"# Follow: rtk proxy tail -f {shlex.quote(str(log))}\n")
print("Each launched run uses up to 4 Docker workers. Start sequentially unless resources permit more.")

### Resume an existing run

Stop the old process first; the run lock prevents concurrent writers. Resume uses
saved search settings, candidates, elites and cached scores. Only unfinished work
and future episodes use the new timeout. The original manifest/source stay intact;
`attempts.json` records changes and `resumes/` retains previous reports.

Set `RESUME_RUN` below and copy the printed command into another terminal. This
cell only prints a command. It does not stop or start any process. For a larger
total generation budget, add `--generations N` to that command.


In [ ]:
RESUME_RUN = None  # For example: RUNS / "pilot-Pendulum-v1-0"
RESUME_EPISODE_TIMEOUT = None  # None inherits the previous attempt's episode limit.

if RESUME_RUN is not None:
    resume_path = Path(RESUME_RUN).resolve()
    command = [str(PYTHON), "-u", "-B", "-m", "examples.elitelist_papers.run",
               "--resume", str(resume_path)]
    if RESUME_EPISODE_TIMEOUT is not None:
        command += ["--episode-timeout", str(RESUME_EPISODE_TIMEOUT)]
    log = WORK / "logs" / f"{resume_path.name}.log"
    shell = f"nohup {shlex.join(command)} >> {shlex.quote(str(log))} 2>&1 < /dev/null & echo $!"
    print(f"In your other terminal: cd {shlex.quote(str(ROOT))}")
    print(f"rtk proxy mkdir -p {shlex.quote(str(WORK / 'logs'))}")
    print("rtk proxy sh -c " + shlex.quote(shell))
else:
    print("Set RESUME_RUN to an existing run directory to print its resume command.")


## 3. Load outputs and audit completeness

This section reads saved JSON/CSV and opens checkpoint databases read-only. It
never opens a simulator or executes a saved policy. The per-run progress figure
includes completed generations from interrupted runs, labelled with run status.
Only completed runs with exported curves enter the later pooled analysis. Failure at held-out evaluation remains a missing
value, not a zero. Excluding a failed search can bias results: report the inventory
and investigate failures before making claims.

Change `RUN_GLOB` to select one experiment group. The loader refuses to pool
incompatible settings within an environment. Review source snapshots as well,
especially when running against an uncommitted working tree.

The loader uses the latest attempt's generation budget and execution settings.
Runs whose policy/episode timeouts changed are listed but excluded from pooled
analysis by default: their cached scores were produced under different limits.
Set `INCLUDE_CHANGED_TIMEOUTS = True` explicitly to inspect such runs.


In [ ]:
from hashlib import sha256

RUN_GLOB = "pilot-*"  # Inspect pilots now; use "main-v1-*" for the new study.
INCLUDE_CHANGED_TIMEOUTS = False  # Opt in explicitly for exploratory analysis of resumed runs.
INCLUDE_CHANGED_CONTEXTS = False  # Context changes also mix search protocols.

def load_runs(root, pattern):
    records = []
    for path in sorted(root.glob(pattern)):
        if not path.is_dir():
            continue
        def read(name, default):
            file = path / name
            return json.loads(file.read_text()) if file.exists() else default
        experiment = read("experiment.json", {})
        attempts = read("attempts.json", [])
        initial_timeouts = (experiment.get("policy_timeout"), experiment.get("episode_timeout", 60))
        timeout_history = [initial_timeouts]
        context_history = [sha256(experiment.get("instructions", "").encode()).hexdigest()]
        for attempt in attempts:
            digest = attempt.get("context_sha256", context_history[-1])
            if digest != context_history[-1]:
                context_history.append(digest)
            limits = (attempt.get("policy_timeout"), attempt.get("episode_timeout", initial_timeouts[1]))
            if limits != timeout_history[-1]:
                timeout_history.append(limits)
        if attempts:
            experiment = dict(experiment)
            experiment["policy_timeout"] = attempts[-1].get("policy_timeout")
            for key in ("policy_timeout", "episode_timeout", "generations", "concurrency", "generation_concurrency", "config"):
                if key in attempts[-1]:
                    experiment[key] = attempts[-1][key]
        status = read("status.json", {"status": "unknown"})
        curves = []
        if (path / "curves.csv").exists():
            for row in csv.DictReader(io.StringIO((path / "curves.csv").read_text())):
                for key in ("generation", "attempts", "calls", "repairs", "discarded", "heldout_n"):
                    row[key] = int(row[key])
                for key in ("search_mean", "heldout_mean"):
                    row[key] = float(row[key]) if row[key] else np.nan
                curves.append(row)
        records.append(dict(path=path, experiment=experiment, status=status, curves=curves,
                            attempts=attempts, timeout_history=timeout_history, context_history=context_history,
                            heldout=read("heldout.json", {}), summary=read("summary.json", {})))
    return records

records = load_runs(RUNS, RUN_GLOB)
for r in records:
    bad = sum(bool(row["heldout_error"]) for row in r["curves"])
    print(f"{r['path'].name}: {r['status']['status']}; generations={len(r['curves'])}; held-out failures={bad}; stop={r['summary'].get('reason', 'unknown')}")
    if len(r["timeout_history"]) > 1:
        print("  Timeout changes:", r["timeout_history"], "(excluded unless INCLUDE_CHANGED_TIMEOUTS=True)")
    if len(r["context_history"]) > 1:
        print("  LLM context changed on resume (excluded unless INCLUDE_CHANGED_CONTEXTS=True)")
    if r["status"].get("error"):
        print("  " + r["status"]["error"])
if not records:
    print("No runs yet. Run a printed command separately, then rerun from this cell.")



### Target attainment and best-policy progress — individual runs

This is the main demonstration for Paper 1. Each panel shows one run, its best
search score at each completed generation, the recorded target (where defined),
and available held-out scores for those same policies. Markers keep generation-one
successes visible. Interrupted runs end at their last completed generation and
remain explicitly labelled; no values are extrapolated or pooled across runs.

The table separates reaching the search target from reaching it on the full
held-out panel. Missing evidence and undefined targets are shown as “—”. For an
unfinished run, “latest” is an observed checkpoint, not a final experimental result.
“Improvement” means latest minus generation one; all environments maximize reward.

A saved target is the run's operational criterion, not a universal definition of
solving the environment. Review source snapshots for changes made during pilots.


In [ ]:
# Per-run evidence: include completed generations even when reporting was interrupted.
progress = []
for r in records:
    trajectory = [dict(row) for row in r["curves"]]
    database = r["path"] / "run.sqlite"
    if database.exists():
        with closing(sqlite3.connect(database.resolve().as_uri() + "?mode=ro", uri=True)) as db:
            db.execute("BEGIN")  # Read both tables from one consistent checkpoint snapshot.
            tables = {row[0] for row in db.execute("SELECT name FROM sqlite_master WHERE type='table'")}
            if {"elitesearch_generation", "elitesearch_organism"} <= tables:
                organisms = {row[0]: row[1:] for row in db.execute(
                    "SELECT id, policy_id, score FROM elitesearch_organism")}
                trajectory = []
                for number, ids in db.execute(
                    "SELECT number, elite_ids FROM elitesearch_generation WHERE status='completed' ORDER BY number"
                ):
                    ids = json.loads(ids)
                    policy_id, score = organisms[ids[0]] if ids else (None, None)
                    trajectory.append(dict(generation=number, policy_id=policy_id, search_mean=score))
    for row in trajectory:
        if row["search_mean"] is not None and not math.isfinite(row["search_mean"]):
            row["search_mean"] = None
        # Match by policy identity; stale summaries must not label a later winner.
        evidence = r["heldout"].get(row["policy_id"], {})
        if not evidence and row["policy_id"] and row["policy_id"] == r["summary"].get("best_id"):
            evidence = r["summary"].get("heldout") or {}
        scores = evidence.get("scores", {})
        complete = set(map(int, scores)) == set(r["experiment"].get("heldout_seeds", []))
        row["heldout_mean"] = fmean(scores.values()) if scores and complete and not evidence.get("failure") else None
    target = r["experiment"].get("reporting_target", r["experiment"].get("config", {}).get("target_score"))
    first = trajectory[0].get("search_mean") if trajectory else None
    latest = trajectory[-1].get("search_mean") if trajectory else None
    heldout = trajectory[-1].get("heldout_mean") if trajectory else None
    hits = [row["generation"] for row in trajectory
            if target is not None and row["search_mean"] is not None and row["search_mean"] >= target]
    progress.append(dict(
        run=r["path"].name, environment=r["experiment"].get("env", "unknown"),
        status=r["status"]["status"], generations=len(trajectory), target=target,
        first_search=first, latest_search=latest,
        search_improvement=latest-first if first is not None and latest is not None else None,
        search_target_reached=bool(hits) if target is not None and latest is not None else None,
        generation_to_target=hits[0] if hits else None,
        latest_heldout=heldout,
        heldout_target_reached=heldout >= target if target is not None and heldout is not None else None,
        trajectory=trajectory,
    ))

if progress:
    FIGURES.mkdir(exist_ok=True)
    fields = [key for key in progress[0] if key != "trajectory"]
    with (FIGURES / "target-and-progress.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(progress)
    def format_progress(value):
        return "—" if value is None else f"{value:.2f}" if isinstance(value, float) else str(value)
    display(Markdown("| " + " | ".join(fields) + " |\n| " + " | ".join(["---"]*len(fields)) + " |\n" +
        "\n".join("| " + " | ".join(format_progress(row[key]) for key in fields) + " |" for row in progress)))
else:
    print("No saved runs to summarize.")


In [ ]:
if progress:
    columns = min(3, len(progress))
    figure, axes = plt.subplots(math.ceil(len(progress)/columns), columns,
        figsize=(5*columns, 3.7*math.ceil(len(progress)/columns)), squeeze=False)
    for ax, result in zip(axes.flat, progress):
        rows = result["trajectory"]
        generations = [row["generation"] for row in rows]
        search = [row["search_mean"] if row["search_mean"] is not None else np.nan for row in rows]
        heldout = [row["heldout_mean"] if row["heldout_mean"] is not None else np.nan for row in rows]
        ax.plot(generations, search, "o-", color="#176b9c", label="Best search mean")
        if any(np.isfinite(heldout)):
            ax.plot(generations, heldout, "s--", color="#ab6226", label="Same policy: held-out")
        if result["target"] is not None:
            ax.axhline(result["target"], color="#398248", linestyle=":", label=f"Target {result['target']:g}")
        if not rows:
            ax.text(0.5, 0.5, "No completed generations", transform=ax.transAxes, ha="center")
        ax.set(title=f"{result['run']}\n{result['status']} · {len(rows)} completed generations",
               xlabel="Generation", ylabel="Mean episode reward")
        ax.set_xticks(generations)
        ax.legend(fontsize=8)
    for ax in list(axes.flat)[len(progress):]:
        ax.set_visible(False)
    figure.tight_layout()
    for extension in ("pdf", "png"):
        figure.savefig(FIGURES / f"target-and-progress.{extension}", bbox_inches="tight")
    plt.show()


In [ ]:
completed = [r for r in records if r["status"]["status"] == "completed" and r["curves"]
             and (INCLUDE_CHANGED_TIMEOUTS or len(r["timeout_history"]) == 1)
             and (INCLUDE_CHANGED_CONTEXTS or len(r["context_history"]) == 1)]
groups = defaultdict(list)
for r in completed:
    maximum = r["experiment"]["generations"]
    actual = r["summary"].get("generations", maximum)
    if not 1 <= actual <= maximum or [row["generation"] for row in r["curves"]] != list(range(1, actual + 1)):
        raise ValueError(f"Completed run has incomplete curves: {r['path']}")
    if actual < maximum and r["summary"].get("reason") != "target_reached":
        raise ValueError(f"Short run has no recorded early-stop reason: {r['path']}")
    groups[r["experiment"]["env"]].append(r)
for env, runs in groups.items():
    signatures = set()
    replicate_ids = []
    for r in runs:
        e = r["experiment"]
        signature = {k: e[k] for k in ("model", "config", "environment", "seeds", "heldout_seeds", "max_output_tokens", "episode_timeout")}
        signature["policy_timeout"] = e.get("policy_timeout")
        signature["reporting_target"] = e.get("reporting_target", e["config"].get("target_score"))
        signature["timeout_history"] = r["timeout_history"]
        signature["context_history"] = r["context_history"]
        signature["gymnasium"] = e["versions"].get("gymnasium")
        signatures.add(json.dumps(signature, sort_keys=True))
        replicate_ids.append(e["search_seed"])
    if len(signatures) != 1:
        raise ValueError(f"Mixed protocols for {env}; narrow RUN_GLOB before pooling.")
    if len(set(replicate_ids)) != len(replicate_ids):
        raise ValueError(f"Duplicate search replicate IDs for {env}; select a single experiment group.")
print(f"{len(completed)} complete runs across {len(groups)} environments.")

## 4. Learning curves

Solid: held-out return of the generation's **search-selected** winner. Dashed:
search return. Shading: interquartile range across independent searches, **not a
confidence interval**. The dotted random-action reference uses the same held-out
panel. Curves retain generation-one saturation and later declines.

Any missing evaluations are reported below the plot. Bands based on few surviving
runs are descriptive; they cannot establish reliable improvement. Curves are never
aggregated across environments with different reward scales.

Early-stopped trajectories end at their observed stopping generation; no scores
are carried forward. Later points may include fewer searches and are conditioned
on not having stopped. Read the per-generation counts before comparing curves.


In [ ]:
if groups:
    FIGURES.mkdir(exist_ok=True)
    columns = min(3, len(groups))
    figure, axes = plt.subplots(math.ceil(len(groups) / columns), columns,
                                figsize=(5 * columns, 3.7 * math.ceil(len(groups) / columns)), squeeze=False)
    for ax, (env, runs) in zip(axes.flat, sorted(groups.items())):
        generations = sorted({row["generation"] for r in runs for row in r["curves"]})
        for metric, label, style, color in (
            ("search_mean", "Search", "--", "#888888"),
            ("heldout_mean", "Held-out", "-", "#176b9c"),
        ):
            matrix = np.array([[next((row[metric] for row in r["curves"] if row["generation"] == g), np.nan)
                                for g in generations] for r in runs])
            quantiles = np.full((3, len(generations)), np.nan)
            for j in range(len(generations)):
                values = matrix[:, j][np.isfinite(matrix[:, j])]
                if len(values):
                    quantiles[:, j] = np.percentile(values, [25, 50, 75])
            ax.plot(generations, quantiles[1], style, color=color, label=label)
            if metric == "heldout_mean":
                ax.fill_between(generations, quantiles[0], quantiles[2], color=color, alpha=0.18)
                counts = np.isfinite(matrix).sum(axis=0)
                print(f"{env}: valid held-out runs by generation = {counts.tolist()} / {len(runs)}")
        random_means = [fmean(r["heldout"]["random"]["scores"].values()) for r in runs
                        if r["heldout"].get("random", {}).get("scores")]
        if random_means:
            ax.axhline(np.median(random_means), linestyle=":", color="#ab6226", label="Random action")
        ax.set(title=f"{env} (n={len(runs)} searches)", xlabel="Generation", ylabel="Episode return")
        ax.legend(fontsize=8)
    for ax in list(axes.flat)[len(groups):]:
        ax.set_visible(False)
    figure.tight_layout()
    figure.savefig(FIGURES / "learning-curves.pdf", bbox_inches="tight")
    figure.savefig(FIGURES / "learning-curves.png", bbox_inches="tight")
    plt.show()
else:
    print("Learning curves will appear after a completed run.")

## 5. Final results and within-search improvement

Each difference pairs the final generation with generation one of the **same
search**. Do not treat candidates or the 100 test episodes as 100 independent
searches. Medians below summarize searches; incomplete held-out pairs remain
counted as failures. Add a preregistered inference procedure once the main-study
sample size is fixed. This descriptive pilot table makes no significance claims.

For early-stopped runs, final means the stopping generation, so final scores can
use different actual budgets. The table also reports targets reached and mean
generations used.


In [ ]:
results = []
for env, runs in sorted(groups.items()):
    first = np.array([r["curves"][0]["heldout_mean"] for r in runs])
    final = np.array([r["curves"][-1]["heldout_mean"] for r in runs])
    valid = np.isfinite(first) & np.isfinite(final)
    delta = final[valid] - first[valid]
    valid_final = final[np.isfinite(final)]
    results.append(dict(
        environment=env, searches=len(runs),
        target_reached=sum(bool(p["search_target_reached"]) for p in progress
                           if p["run"] in {r["path"].name for r in runs}),
        mean_generations=fmean(len(r["curves"]) for r in runs), valid_final=len(valid_final), paired_searches=int(valid.sum()),
        final_median=float(np.median(valid_final)) if len(valid_final) else None,
        improvement_median=float(np.median(delta)) if len(delta) else None,
        improved_searches=int((delta > 0).sum()),
        heldout_failed_generations=sum(bool(row["heldout_error"]) for r in runs for row in r["curves"]),
        discarded_attempts=sum(r["curves"][-1]["discarded"] for r in runs),
    ))
if results:
    fields = list(results[0])
    display(Markdown("| " + " | ".join(fields) + " |\n| " + " | ".join(["---"] * len(fields)) + " |\n" +
                     "\n".join("| " + " | ".join(str(row[k]) for k in fields) + " |" for row in results)))
    with (FIGURES / "results.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(results)
else:
    print("No completed results to summarize.")

## 6. Resource accounting

Candidate attempts and model calls are different: repairs add calls. Native API
usage/cost is retained when returned. Missing billing data is **unknown**, not free.
Reconcile with OpenRouter before publication. Latencies from overlapping model
calls must not be summed and described as wall-clock duration. End-to-end elapsed
time includes setup and held-out reporting. The current evaluator does not persist
environment-step counts, so do not claim exact step efficiency from these outputs.

In [ ]:
for r in records:
    path = r["path"] / "llm_calls.jsonl"
    calls = []
    if path.exists():
        lines = path.read_text().splitlines()
        for index, line in enumerate(lines):
            try:
                calls.append(json.loads(line))
            except json.JSONDecodeError:
                if index != len(lines) - 1:
                    raise
                print(f"{r['path'].name}: incomplete final API log line; accounting is partial")
    totals = {}
    for field in ("prompt_tokens", "completion_tokens", "cost"):
        values = [(call.get("usage") or {}).get(field) for call in calls]
        totals[field] = sum(values) if values and all(v is not None for v in values) else "unknown"
    durations = [a.get("elapsed_seconds") for a in (r["attempts"] or [r["status"]])]
    elapsed = sum(durations) if all(v is not None for v in durations) else "unknown/incomplete"
    providers = sorted({call["provider"] for call in calls if call.get("provider")})
    print(r["path"].name, {"logged_calls": len(calls), **totals,
                          "providers": providers or "unknown", "elapsed_seconds": elapsed})

## 7. Inspect a policy and its ancestry

Choose the run and generation explicitly. This cell reads saved source as text;
**it does not execute generated code in the notebook**. Use the existing sandbox
for any future evaluation. Describe concrete changes in a successful, typical and
failed trajectory, and explain how those examples were selected.

In [ ]:
INSPECT_RUN = completed[0]["path"] if completed else None  # Replace with a Path to another run.
INSPECT_GENERATION = 1
if INSPECT_RUN is not None:
    selected = next(r for r in completed if r["path"] == INSPECT_RUN)
    row = next((row for row in selected["curves"] if row["generation"] == INSPECT_GENERATION), None)
    if row and row["policy_id"]:
        connection = sqlite3.connect((INSPECT_RUN / "run.sqlite").resolve().as_uri() + "?mode=ro", uri=True)
        try:
            policy = connection.execute(
                "SELECT name, description, implementation, parent_ids, kind FROM elitesearch_organism WHERE policy_id = ?",
                (row["policy_id"],),
            ).fetchone()
        finally:
            connection.close()
        if policy:
            print(f"{policy[0]} | origin={policy[4]} | parents={policy[3]}")
            print(policy[1])
            display(Code(policy[2], language="python"))
    else:
        print("No valid winner for that generation.")
else:
    print("Run a simulation first, then select a saved policy here.")

## 8. Research notes → manuscript

Use the target-and-progress table and figures above to support each observation.
Paper 1 demonstrates target attainment or score improvement in the observed runs;
it does not require an algorithm-comparison or ablation study.

Best-so-far search scores cannot decrease because elites are retained. The evidence
for improvement is a positive increase, not merely a nondecreasing curve. A flat
curve that starts above target is an immediate success. A flat curve below target
is a plateau. Tasks with no recorded target can support improvement claims, but
not a target-based solved claim. Held-out curves need not be monotonic.

- **Protocol frozen:** record date, run group, source snapshot, model/provider and budgets.
- **Success case:** what changed in the policy, and did held-out reward improve?
- **Typical case:** how was “typical” chosen without picking a flattering example?
- **Failure case:** stagnation, invalid programs, timeouts, seed overfitting or insufficient budget?
- **BipedalWalker:** distinguish low reward, partial walking and your prespecified solved criterion.
- **Saturation:** which tasks were already solved in generation one?
- **Uncertainty:** number of independent searches; missing results; noisy episode returns.
- **Outstanding evidence:** billing reconciliation; frozen reproduction artifact; full main-study runs.
- **Follow-up only:** independent-sampling control, ablations and broad comparisons.

Before submission, include pseudocode, full prompts, code snapshots, per-run data,
environment settings, limitations and the rule used to select illustrative policies.
Keep ablation hypotheses in Paper 2 and broad algorithm comparisons in Paper 3.

Sources: [Gymnasium](https://gymnasium.farama.org/),
[OpenRouter Nitro](https://openrouter.ai/docs/guides/routing/model-variants/nitro).